# 07 - Governed generation and evaluation

Team One Sheep Army. Runs every arm on the held-out test photos and scores them with the five metric families of the brief plus per-attribute adherence.

| arm | what it is |
|---|---|
| `fixed` | constant weights (old baseline) |
| `scheduled` | hand schedule (old baseline; the router was statistically indistinguishable from it) |
| `governed` | gates predicted by the governance network |
| `governed_oracle` | gates from the label rules (upper bound on detection) |
| `governed_nosplit` | gates predicted, but the reference embedding is not split (ablation) |
| `governed_lora` | `governed` + the fusion LoRA |

Needs `governance.pt` (notebook 05) and, for the last arm, `fusion_lora.safetensors` (notebook 06). Resumable: existing images are skipped.

In [ ]:
# ---- CONFIG + SETUP (same in notebooks 05, 06, 07) ----
DRIVE_ROOT = "/content/drive/MyDrive/sketchy_dataset_one_sketch"
ZIP_PATH = f"{DRIVE_ROOT}/pilot_v1.zip"
LOCAL_DATA = "/content/pilot_v1"
GOV_DIR = f"{DRIVE_ROOT}/governance"            # governance.pt, cached features, fusion LoRA all live here
# The code lives in the repo's src/ folder (cloned below). To use a copy on Drive instead, set REPO_DIR to it.
REPO_URL = "https://github.com/Harshitha-282007/IE643_final.git"
REPO_BRANCH = "governed-pipeline"                # change to "master" once the branch is merged
REPO_DIR = "/content/repo"

import os, sys, glob, json, zipfile
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
from google.colab import drive
drive.mount("/content/drive")
if not os.path.isdir(REPO_DIR):
    !git clone -q -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}
assert os.path.exists(f"{REPO_DIR}/src/pipeline.py"), "src/pipeline.py not found: push the new code or point REPO_DIR at it"
sys.path.insert(0, f"{REPO_DIR}/src")
os.makedirs(GOV_DIR, exist_ok=True)
if not glob.glob(f"{LOCAL_DATA}/**/metadata.jsonl", recursive=True):
    os.makedirs(LOCAL_DATA, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(LOCAL_DATA)
DATA_DIR = os.path.dirname(glob.glob(f"{LOCAL_DATA}/**/metadata.jsonl", recursive=True)[0])
BASE_ROWS = [json.loads(l) for l in open(f"{DATA_DIR}/metadata.jsonl") if l.strip()]
print("data:", DATA_DIR, "| base rows:", len(BASE_ROWS))

In [ ]:
!pip uninstall -y torchao -q
!pip install -q -U diffusers transformers accelerate safetensors peft lpips opencv-python-headless
import torch, numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
from governance import labels as L, model as GM, metrics as MT
from governance.evalkit import EvalKit
from pipeline import Pipeline, seed_for

RUN = "governed_v1"
OUT = f"{DRIVE_ROOT}/sdxl_runs/{RUN}"
RES, STEPS, SEEDS = 640, 20, [0]
MAX_PHOTOS = None                    # e.g. 6 for a quick dry run; None = all 20 test photos
LORA = f"{GOV_DIR}/fusion_lora/fusion_lora.safetensors"
os.makedirs(OUT, exist_ok=True)

# one variant of each kind per test photo; rule labels attached
rows = [r for r in L.expand_rows(BASE_ROWS, per_type=1, seed=1, kinds=("obj", "col", "mat", "ref")) if r["split"] == "test"]
if MAX_PHOTOS:
    keep = sorted({r["original_id"] for r in rows})[:MAX_PHOTOS]
    rows = [r for r in rows if r["original_id"] in keep]
kind = lambda r: "base" if r["id"] == r["original_id"] else r["id"].split("_")[-1][:3]
print(len(rows), "items from", len({r["original_id"] for r in rows}), "base photos:", pd.Series([kind(r) for r in rows]).value_counts().to_dict())

In [ ]:
# ---- models ----
P = Pipeline(res=RES, steps=STEPS, lora=LORA)
net = GM.load(f"{GOV_DIR}/governance.pt")
op = lambda r, k: Image.open(f"{DATA_DIR}/{r[k]}")
ITEMS = {}
for i in tqdm(range(0, len(rows), 16), desc="encode"):
    chunk = rows[i:i + 16]
    for r, it in zip(chunk, P.make_items([dict(key=r["id"], sketch=op(r, "sketch"), ref=op(r, "reference"), caption=r["caption"],
                                               obj=r.get("caption_category", r["category"]), seed=seed_for(r["original_id"])) for r in chunk])):
        it["gates_pred"], it["gates_rule"] = GM.predict_gates(net, it["x"]), {k: float(v) for k, v in r["gates"].items()}
        ITEMS[r["id"]] = it
g = pd.DataFrame([dict(kind=kind(r), **ITEMS[r["id"]]["gates_pred"]) for r in rows])
print("predicted gates by kind (obj rows should have g_obj high, col rows g_col high, mat rows g_mat high):")
print(g.groupby("kind").mean().round(2))

In [ ]:
# ---- sanity checks before the long run: every setting must visibly do something ----
r0 = next(r for r in rows if kind(r) == "col"); it0 = ITEMS[r0["id"]]
from governance.evalkit import thin_mask
def obj_colours(img):
    reg = MT.object_region(thin_mask(it0["sketch"]))
    return MT.name_hist(MT.foreground_pixels(np.array(img.convert("RGB").resize((256, 256))), reg))
torch.cuda.reset_peak_memory_stats(); P.use_lora(False)
shots = [("reference governs colour (g_col=0)", dict(gates=dict(g_obj=0, g_col=0, g_mat=0))),
         ("text governs colour (g_col=1)", dict(gates=dict(g_obj=0, g_col=1, g_mat=0))),
         ("g_col=0, style block OFF (plain scale)", dict(gates=dict(g_obj=0, g_col=0, g_mat=0), style_only=False)),
         ("legacy scheduled", dict(shaped=True))]
fig, ax = plt.subplots(1, 2 + len(shots), figsize=(3.2 * (2 + len(shots)), 3.6))
ax[0].imshow(it0["sketch"]); ax[0].set_title("sketch", fontsize=8); ax[1].imshow(it0["ref"]); ax[1].set_title(f"ref: {r0['ref_colors']}", fontsize=8)
imgs = []
for a, (name, kw) in zip(ax[2:], shots):
    im = P.run(it0, **kw); imgs.append(im); a.imshow(im)
    top = sorted(obj_colours(im).items(), key=lambda kv: -kv[1])[:2]
    a.set_title(f"{name}\n{[(n, round(s, 2)) for n, s in top]}", fontsize=7)
for a in ax: a.axis("off")
plt.suptitle(r0["caption"], fontsize=9); plt.show()
d = lambda a, b: float(np.abs(np.asarray(a, float) - np.asarray(b, float)).mean())
print(f"mean pixel difference  g_col 0 vs 1: {d(imgs[0], imgs[1]):.1f} | style block vs plain scale: {d(imgs[0], imgs[2]):.1f}  (both must be well above 0)")
print("peak GPU GB:", round(torch.cuda.max_memory_allocated() / 1e9, 2))

In [ ]:
# ---- generate every arm (resumable) ----
ARMS = {"fixed": dict(shaped=False), "scheduled": dict(shaped=True),
        "governed": dict(g="pred"), "governed_oracle": dict(g="rule"), "governed_nosplit": dict(g="pred", split=False)}
if P.has_lora:
    ARMS["governed_lora"] = dict(g="pred", lora=True)
manifest = []
for r in tqdm(rows, desc="generate"):
    it = ITEMS[r["id"]]
    for s in SEEDS:
        for arm, a in ARMS.items():
            path = f"{OUT}/eval/{arm}/{r['id']}_s{s}.png"
            if not os.path.exists(path):
                os.makedirs(os.path.dirname(path), exist_ok=True)
                P.use_lora(a.get("lora", False))
                kw = dict(shaped=a["shaped"]) if "g" not in a else dict(gates=it[f"gates_{a['g']}"], split=a.get("split", True))
                P.run(it, seed_off=s, **kw).save(path)
            manifest.append(dict(id=r["id"], base=r["original_id"], kind=kind(r), arm=arm, seed=s, generated=path, caption=r["caption"],
                                 **{f"pred_{k}": v for k, v in it["gates_pred"].items()}, **r["gates"]))
    torch.cuda.empty_cache()
pd.DataFrame(manifest).to_csv(f"{OUT}/manifest.csv", index=False)
print(len(manifest), "images | peak GPU GB:", round(torch.cuda.max_memory_allocated() / 1e9, 2))

In [ ]:
# ---- score (resumable) ----
EK = EvalKit()
HP = f"{OUT}/scores.csv"
done = pd.read_csv(HP) if os.path.exists(HP) else pd.DataFrame()
have = set(zip(done.id, done.arm, done.seed)) if len(done) else set()
recs = done.to_dict("records") if len(done) else []
by_id = {r["id"]: r for r in rows}
for m in tqdm(manifest, desc="score"):
    if (m["id"], m["arm"], m["seed"]) in have: continue
    r, it = by_id[m["id"]], ITEMS[m["id"]]
    tgt = op(r, "target") if m["kind"] == "base" else None          # the real photo is the answer only for unedited rows
    recs.append(dict(id=m["id"], base=m["base"], kind=m["kind"], arm=m["arm"], seed=m["seed"],
                     **EK.score(Image.open(m["generated"]).convert("RGB"), r, it["sketch"], it["ref"], tgt)))
S = pd.DataFrame(recs); S.to_csv(HP, index=False)
print("scored", len(S))

In [ ]:
# ---- results: the brief's five metric families + adherence, per arm ----
num = S.select_dtypes("number").columns.drop("seed", errors="ignore")
Si = S.groupby(["id", "base", "kind", "arm"], as_index=False)[list(num)].mean()       # average seeds inside each item first
order = list(ARMS)
FAM = {"text-image alignment": ["clip_t", "sem_margin", "obj_hit"],
       "sketch consistency": ["edge_recall", "edge_precision", "edge_iou"],
       "reference similarity": ["dino_ref", "attr_B", "col_share", "col_hit", "col_leak", "mat_hit"],
       "layout consistency": ["layout_sketch_iou", "layout_tgt_iou"],
       "perceptual quality (unedited rows)": ["lpips_tgt", "dino_tgt"]}
for fam, cols in FAM.items():
    cols = [c for c in cols if c in Si]
    print(f"\n== {fam} ==  (mean over items)"); print(Si.groupby("arm")[cols].mean().reindex(order).round(3))
print("\n== adherence by kind: did the output follow the governing modality? ==")
print(Si.pivot_table(index="kind", columns="arm", values="col_hit").reindex(columns=order).round(2).rename_axis("colour hit rate"))
print(Si.pivot_table(index="kind", columns="arm", values="obj_hit").reindex(columns=order).round(2).rename_axis("object hit rate"))
Si.to_csv(f"{OUT}/results_per_item.csv", index=False)
Si.groupby("arm")[[c for cs in FAM.values() for c in cs if c in Si]].mean().reindex(order).round(4).to_csv(f"{OUT}/results_table.csv")

In [ ]:
# ---- paired comparisons, bootstrap over BASE PHOTOS (conflict variants share a photo) ----
PAIRS = [("governed", "scheduled"), ("governed", "fixed"), ("governed", "governed_nosplit"), ("governed_oracle", "governed")]
if "governed_lora" in ARMS: PAIRS += [("governed_lora", "governed"), ("governed_lora", "fixed")]
METRICS = ["col_share", "col_hit", "col_leak", "mat_hit", "obj_hit", "sem_margin", "clip_t", "edge_precision", "edge_iou",
           "dino_ref", "layout_sketch_iou", "lpips_tgt", "dino_tgt"]
out = []
for a, b in PAIRS:
    A, B = Si[Si.arm == a].set_index("id"), Si[Si.arm == b].set_index("id")
    ks = A.index.intersection(B.index)
    for m in [m for m in METRICS if m in Si]:
        bs = MT.paired_bootstrap((A.loc[ks, m] - B.loc[ks, m]).values, A.loc[ks, "base"].values, n=5000)
        if bs["n"] >= 3:
            out.append(dict(comparison=f"{a} - {b}", metric=m, **{k: round(v, 4) if isinstance(v, float) else v for k, v in bs.items()},
                            significant="yes" if bs["lo"] > 0 or bs["hi"] < 0 else "no"))
PT = pd.DataFrame(out); PT.to_csv(f"{OUT}/paired_comparisons.csv", index=False)
print(PT.to_string(index=False))
print("\nA difference counts only if the interval excludes zero. lpips_tgt and col_leak: lower is better.")

In [ ]:
# ---- contact sheet: one item of each kind, every arm ----
show = [next(r for r in rows if kind(r) == k) for k in ["base", "obj", "col", "mat", "ref"] if any(kind(r) == k for r in rows)]
fig, ax = plt.subplots(len(show), 2 + len(ARMS), figsize=(3 * (2 + len(ARMS)), 3 * len(show)))
for i, r in enumerate(show):
    it = ITEMS[r["id"]]
    ax[i, 0].imshow(it["sketch"]); ax[i, 0].set_title(f"{kind(r)}: {r['caption'][:36]}", fontsize=7)
    ax[i, 1].imshow(it["ref"]); ax[i, 1].set_title(f"ref: {r['ref_colors']}, {r['ref_material']}", fontsize=7)
    for j, arm in enumerate(ARMS):
        ax[i, 2 + j].imshow(Image.open(f"{OUT}/eval/{arm}/{r['id']}_s{SEEDS[0]}.png"))
        ax[i, 2 + j].set_title(arm if i == 0 else "", fontsize=9)
    gp = it["gates_pred"]; ax[i, 2].set_xlabel(f"obj {gp['g_obj']:.2f} col {gp['g_col']:.2f} mat {gp['g_mat']:.2f}", fontsize=7)
for a in ax.ravel(): a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.savefig(f"{OUT}/contact_sheet.png", dpi=110); plt.show()
print("saved to", OUT)